# One scan box for the campaign, and what the scaled profiles look like

The fits are settled next door (`mtanh_fit_quality/`). This notebook takes them,
applies the campaign's four axes through `DischargePhysics.apply_mtanh_full`, and
checks every profile that would be handed to CHEASE-BS — at the box edges and at
intermediate steps — to answer one question: **are these reasonable plasmas?**

Both notebooks import `pedestal_scan.py` at the repo root, which owns the
discharge list, the fit settings, the pedestal rule, the axes and the bounds, so
the numbers here and the verdict there cannot drift apart.

**One box, not four.** The sparse grid's axes are common to the campaign and the
sampler chooses its own points, so per-discharge bounds cannot be handed to it.
The box is therefore the ±30% ceiling, shrunk on an axis only where some
discharge stops being *physical* — the intersection across discharges, not an
intersection with Boyle. That keeps the number of control points minimal: four
axes, one interval each.

**Boyle is reporting, not a constraint.** Earlier versions cut each axis back to
the scale factors that would land that discharge inside Boyle's band. That was
the wrong shape of answer for a sampler that explores on its own — and it
dropped axes that were physically fine. Where each discharge sits relative to
the bands is still reported below; it just no longer decides the box.

**Four axes, electron channel only.** $T_{e,\text{ped}}$, $n_{e,\text{ped}}$,
$\Delta T_e$, $\Delta n_e$. The KBM drive is $\nabla p_e$, which those four set
between them, and Boyle 2011 quotes exactly these quantities. The ion channel is
not unscanned: scaling $n_e$ rewrites $n_i$ and $n_z$ through quasineutrality
inside `DischargePhysics`, and $p_e$ is read off the transformed object rather
than reconstructed by hand.

**±30% ceiling.** The survey spec's reference is Hatch's ±10–30% grid around the
experimental pre-ELM state, and nothing beyond ~1.4 has ever been through
cheaseBS — which the next step, cheaseBS on these points, is what settles.

In [ ]:
import sys, json, pathlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pedestal_scan.py").exists())
sys.path.insert(0, str(ROOT))

from pedestal_scan import (Campaign, AXES, SCALE_SANITY, BOYLE_WIDTHS, TARGETS,
                           CORE_FRAC, CORE_RHO, COL, ANALYSIS_RADII, TRUST,
                           window_stats)

camp = Campaign()
shots = camp.shots

# Steps across each axis, edges included. These are check points for this
# notebook, NOT the scan grid -- the sparse sampler picks its own points inside
# the box. An odd number keeps a mid-box point in the set.
N_STEPS = 5

print(f"axes: {', '.join(AXES)}")
print(f"scale ceiling: {SCALE_SANITY}   check points per axis: {N_STEPS}")
print("\nregime per discharge (reported; it no longer constrains the box):")
for s in shots:
    pinned = [v for v in ("Te", "ne") if camp[s].var[v].get("pinned")]
    print(f"  {s}: {camp.regime[s]:<9} ("
          + ", ".join(f"{k} {v:.1f}%" for k, v in camp.widths[s].items()) + ")"
          + (f"   width unreliable: {'/'.join(pinned)} b_pos on bound"
             if pinned else ""))

## First: are the fits still the ones we approved?

Scaling multiplies fit parameters, so a fit that changed underneath this notebook
changes the meaning of every scale factor in it. Same check as
`mtanh_fit_quality/`, in two views so nothing downstream is read on trust: **full
radius** (what `apply_mtanh_full` writes and CHEASE-BS reshapes on) and **pedestal
zoom** (where the axes act).

Each panel prints the rms inside that discharge's verdict window and over the
whole profile. Green is under the 1% trust threshold. A disagreement with the fit
notebook means `pedestal_scan.FIT_KWARGS` moved and everything below is stale.

In [ ]:
FIT_VARS = ("Te", "ne", "pe")


def fit_check(zoom):
    """Data vs fit for the variables the axes act on. zoom=True: pedestal only."""
    fig, ax = plt.subplots(len(FIT_VARS), len(shots),
                           figsize=(3.4 * len(shots), 2.3 * len(FIT_VARS)),
                           squeeze=False)
    for j, shot in enumerate(shots):
        d = camp[shot]
        xl = (d.win[0] - 0.05, 1.0) if zoom else (0.0, 1.0)
        m = (d.x >= xl[0]) & (d.x <= xl[1])
        for i, var in enumerate(FIT_VARS):
            a, f = ax[i][j], d.var[var]
            a.plot(d.x[m], f["y"][m], ".", ms=3, color="0.55", label="data")
            a.plot(d.x[m], f["yhat"][m], "-", lw=1.5, color=COL[var], label="fit")
            r = window_stats(d.x, f["err"], d.win)[0]
            a.text(0.03, 0.08,
                   f"win {r*100:.2f}%  |  all {f['rms_global']*100:.2f}%",
                   fontsize=7, transform=a.transAxes,
                   color="green" if r <= TRUST else "tab:red")
            a.axvspan(*d.win, color="tab:green", alpha=0.10)
            for r0 in ANALYSIS_RADII[shot]:
                a.axvline(r0, color="k", lw=0.8, ls=":", alpha=0.8)
            a.set_xlim(*xl); a.tick_params(labelsize=7)
            if i == 0:
                a.set_title(f"{shot}  {camp.regime[shot]}", fontsize=10)
            if j == 0:
                a.set_ylabel(var)
            if i == len(FIT_VARS) - 1:
                a.set_xlabel("rho_tor")
    ax[0][-1].legend(fontsize=6)
    fig.suptitle("pedestal zoom — shaded: verdict window, dotted: GENE radii"
                 if zoom else
                 "full radius — the profile apply_mtanh_full writes")
    plt.tight_layout()
    return fig


fit_check(zoom=False); plt.show()
fit_check(zoom=True);  plt.show()

## The box

Four axes, one interval each, valid for every discharge. Each axis starts at the
±30% ceiling and is shrunk only where some discharge stops being physical there:
non-positive anywhere, non-monotonic outward through the pedestal, or no longer
re-fittable — since CHEASE-BS and the profile writers refit downstream.

The notes say which discharge limited which axis. No note means the ceiling
itself is the binding constraint, not the physics.

In [ ]:
BOX, NOTES = camp.shared_box()

print("shared scan box (hand this to ScanStudy):")
print(json.dumps({a: list(v) for a, v in BOX.items()}, indent=1))
for n in NOTES:
    print(f"  ! {n}")
if not NOTES:
    print(f"  no axis needed shrinking — the whole {SCALE_SANITY} ceiling is "
          "physical for all four discharges")

# Check points for this notebook. The sparse grid samples the box itself; these
# are only where the profiles below are drawn and measured.
STEPS = {axis: np.linspace(lo, hi, N_STEPS) for axis, (lo, hi) in BOX.items()}
print(f"\ncheck points per axis: "
      + ", ".join(f"{s:.2f}" for s in next(iter(STEPS.values()))))
print(f"{len(shots)} discharges x {len(BOX)} axes x {N_STEPS} points = "
      f"{len(shots) * len(BOX) * N_STEPS} profiles to check, plus "
      f"{len(shots)} nominals")

with open("sg_bounds.json", "w") as fh:
    json.dump({"form": "full", "axes": list(AXES),
               "scale_ceiling": list(SCALE_SANITY),
               "bounds": {a: list(v) for a, v in BOX.items()},
               "shared_across_discharges": True,
               "discharges": [int(s) for s in shots],
               "regime": {str(k): v for k, v in camp.regime.items()},
               "boyle_widths": BOYLE_WIDTHS, "targets": TARGETS,
               "notes": NOTES}, fh, indent=1)
print("\nwritten: sg_bounds.json")

## Where that box puts each discharge, against Boyle

The same box means different physical values per discharge — that is inherent to
scale-factor axes and is why the campaign has to be reported in physical units.
Each panel below is one axis: the horizontal bars are the interval each discharge
reaches across the box, and the shaded band is Boyle's range for that
discharge's regime.

Read it as coverage of the *population*, not of each discharge: between them the
four discharges should span a decent part of each band. A bar sitting entirely
outside its band is a discharge the ±30% box cannot bring into Boyle's observed
range — worth knowing when the results are written up, not a reason to move the
box.

In [ ]:
reach = {}
for shot in shots:
    d = camp[shot]
    for axis, (lo, hi) in BOX.items():
        disp = AXES[axis][4]
        vals = sorted(d.metric(axis, s) * disp for s in (lo, hi))
        reach[(shot, axis)] = vals

hdr = (f"{'axis':<16} {'shot':>7} {'nominal':>9} {'reach across box':>21} "
       f"{'unit':<6} {'Boyle/target':>13}  overlap")
print(hdr); print("-" * len(hdr))
for axis in BOX:
    disp = AXES[axis][4]
    for shot in shots:
        d = camp[shot]
        lo_t, hi_t = camp.target(shot, axis)
        r_lo, r_hi = reach[(shot, axis)]
        ov = max(0.0, min(r_hi, hi_t) - max(r_lo, lo_t)) / (hi_t - lo_t)
        print(f"{axis:<16} {shot:>7} {d.metric(axis) * disp:>9.3f} "
              f"{r_lo:>9.3f}-{r_hi:<11.3f} {AXES[axis][3]:<6} "
              f"{f'{lo_t}-{hi_t}':>13}  {ov:>5.0%}"
              + ("   [fit pinned]" if d.var[AXES[axis][0]].get("pinned") else ""))
    print()

fig, axr = plt.subplots(1, len(BOX), figsize=(3.6 * len(BOX), 3.0), squeeze=False)
for j, axis in enumerate(BOX):
    ax = axr[0][j]
    for i, shot in enumerate(shots):
        y = len(shots) - 1 - i
        lo_t, hi_t = camp.target(shot, axis)
        ax.plot([lo_t, hi_t], [y + 0.22] * 2, "-", lw=7, color="tab:green",
                alpha=0.20, solid_capstyle="butt")
        r_lo, r_hi = reach[(shot, axis)]
        ax.plot([r_lo, r_hi], [y] * 2, "-", lw=6,
                color=COL.get(AXES[axis][0], "0.3"), solid_capstyle="butt")
        ax.plot([camp[shot].metric(axis) * AXES[axis][4]], [y], "*", ms=11,
                color="k", zorder=3)
    ax.set_yticks(range(len(shots)))
    ax.set_yticklabels([str(s) for s in shots[::-1]], fontsize=8)
    ax.set_xlabel(AXES[axis][3]); ax.set_title(axis, fontsize=9)
    ax.grid(axis="x", alpha=0.3); ax.tick_params(labelsize=7)
fig.suptitle("what the shared box reaches per discharge (bar) vs Boyle's band "
             "for its regime (green) — star = nominal")
plt.tight_layout(); plt.show()

## The profiles themselves

One figure per discharge. **A row per axis** — all four are live for every
discharge now — and six columns: $T_e$, $n_e$, derived $p_e$, each at full radius
and again zoomed on the pedestal. One curve per check point, blue at the low edge
to red at the high edge; nominal is the heavy black line.

**Panels showing only the black line** are the other species on a single-axis
row: a `Te_ped_scale` row does not move $n_e$, and vice versa. The $p_e$ column
always moves, since $p_e = n_e T_e$. The one asymmetry that is physical rather
than cosmetic: scaling $n_e$ also rewrites $n_i$ and $n_z$ through
quasineutrality, while scaling $T_e$ leaves $T_i$ alone.

This is the check that matters before submitting, because it is literally the
object CHEASE-BS is handed. Read it for **separation** (curves bunched on nominal
mean an axis that buys nothing), **ordering** (colours should progress with no
crossings), and **the core** — the axes are named after the pedestal, so a large
core change is the transform leaking, since Stefanikova's core Gaussian is
anchored to `a_height` at $r=0$ only.

In [ ]:
GAL_VARS = ("Te", "ne", "pe")
VIEWS = (("full", False), ("ped", True))


def prof(q, var):
    """pe derived, as CHEASE builds it; everything else straight off the object."""
    if var == "pe":
        return (np.asarray(q.ds["ne"].values, dtype=float)
                * np.asarray(q.ds["Te"].values, dtype=float))
    return np.asarray(q.ds[var].values, dtype=float)


SWEEP = {}          # (shot, axis, s) -> transformed DischargePhysics, reused below
cmap = plt.get_cmap("coolwarm")

for shot in shots:
    d = camp[shot]
    axes_live = list(BOX)
    ncol = len(GAL_VARS) * len(VIEWS)
    fig, axg = plt.subplots(len(axes_live), ncol,
                            figsize=(2.7 * ncol, 2.3 * len(axes_live)),
                            squeeze=False)
    for i, axis in enumerate(axes_live):
        scales = STEPS[axis]
        scaled = [SWEEP.setdefault((shot, axis, float(sc)), d.scaled(axis, sc))
                  for sc in scales]
        for c, (view, zoom) in enumerate(VIEWS):
            xl = (d.win[0] - 0.05, 1.0) if zoom else (0.0, 1.0)
            m = (d.x >= xl[0]) & (d.x <= xl[1])
            for v, var in enumerate(GAL_VARS):
                ax = axg[i][c * len(GAL_VARS) + v]
                ax.plot(d.x[m], prof(d.phys, var)[m], "-", lw=2.4, color="k",
                        label="nominal", zorder=4)
                for k, (sc, q) in enumerate(zip(scales, scaled)):
                    ax.plot(d.x[m], prof(q, var)[m], "-", lw=1.1,
                            color=cmap(k / max(len(scales) - 1, 1)),
                            label=f"{sc:.2f}", alpha=0.95)
                ax.axvspan(*d.win, color="tab:green", alpha=0.08)
                for r0 in ANALYSIS_RADII[shot]:
                    ax.axvline(r0, color="k", lw=0.8, ls=":", alpha=0.7)
                ax.set_xlim(*xl); ax.tick_params(labelsize=6)
                if var != "pe" and var != AXES[axis][0]:
                    ax.text(0.5, 0.92, "unchanged by this axis", fontsize=5.5,
                            ha="center", va="top", color="0.45",
                            transform=ax.transAxes)
                if i == 0:
                    ax.set_title(f"{var}  ({view})", fontsize=9)
                if c == 0 and v == 0:
                    ax.set_ylabel(axis.replace("_scale", "") + "\n" + var,
                                  fontsize=8)
                else:
                    ax.set_ylabel(var, fontsize=7)
                if i == len(axes_live) - 1:
                    ax.set_xlabel("rho_tor", fontsize=8)
                if i == 0 and c == 0 and v == len(GAL_VARS) - 1:
                    ax.legend(fontsize=5, ncol=2)
    fig.suptitle(f"{shot} ({camp.regime[shot]}) — check points across the shared "
                 "box, blue = low edge, red = high edge")
    plt.tight_layout(); plt.show()

## Are they reasonable? — every check point, measured

Four tests over all of them, not just the corners.

- **Positive and monotonic** through the pedestal, and **re-fittable** — the
  minimum bar for submitting an equilibrium at all. These are the tests that
  defined the box, so a failure here would mean a bug, not a finding.
- **$p_e$ width against Boyle's $\Delta p_e$ band** (his panel 7g). $p_e$ has no
  axis of its own — it is derived, and CHEASE builds pressure from the profiles
  regardless — so the band is a free consistency read: individually legal $n_e$
  and $T_e$ widths can still combine into a pedestal he never observed. Reported,
  not enforced.
- **Core drift against budget.** Drift is the largest fractional change inside
  $\rho_t<0.5$; budget is 30% of the change the axis produced at its own pedestal
  top. Above the diagonal the knob moved the core more than the pedestal it is
  named after. This is a property of scaling a Stefanikova fit — the height axes
  do it structurally — so it is measured and carried forward, not filtered.

In [ ]:
checks = []
for (shot, axis, s), q in SWEEP.items():
    d = camp[shot]
    var = AXES[axis][0]
    y = np.asarray(q.ds[var].values, dtype=float)
    m = (d.x >= 0.6) & (d.x <= 1.0)
    drift, budget = d.core_drift(axis, s)
    lo_pe, hi_pe = BOYLE_WIDTHS[camp.regime[shot]]["dpe"]
    try:
        w, pe_pos, pe_pinned = d.width_psin(q, "pe", with_flags=True)
        fitted = True
    except Exception:
        w, pe_pos, pe_pinned, fitted = np.nan, np.nan, False, False
    checks.append({"shot": shot, "axis": axis, "s": s, "pe_width": w,
                   "pe_ok": bool(lo_pe <= w <= hi_pe), "fitted": fitted,
                   "pe_pinned": bool(pe_pinned), "core": drift,
                   "budget": budget, "positive": float(np.min(y)) > 0,
                   "monotonic": not np.any(np.diff(y[m])
                                           > 0.02 * float(np.max(y[m])))})

bad_phys = [c for c in checks if not (c["positive"] and c["monotonic"]
                                      and c["fitted"])]
print(f"{len(checks)} check points")
print(f"  non-positive / non-monotonic / unfittable : {len(bad_phys)}"
      + ("  <-- bug: these define the box" if bad_phys else ""))
print(f"  pe width outside Boyle's band            : "
      f"{sum(not c['pe_ok'] for c in checks)}  (reported)")
print(f"  core drift over budget                   : "
      f"{sum(c['core'] > c['budget'] for c in checks)}  (reported)")
print(f"  pe REFIT on the b_pos bound              : "
      f"{sum(c['pe_pinned'] for c in checks)}  (their pe width is an artefact "
      "of the bound, not a measurement)")
for c in bad_phys:
    print(f"    ! {c['shot']} {c['axis']} {c['s']:.2f}")

print(f"\n{'axis':<16} {'shot':>7} {'pe width across box':>21} "
      f"{'Boyle dpe':>10}  {'max core drift':>14}")
print("-" * 74)
for axis in BOX:
    for shot in shots:
        sub = [c for c in checks if c["shot"] == shot and c["axis"] == axis]
        ws = [c["pe_width"] for c in sub]
        band = BOYLE_WIDTHS[camp.regime[shot]]["dpe"]
        worst = max(sub, key=lambda c: c["core"])
        npin = sum(c["pe_pinned"] for c in sub)
        print(f"{axis:<16} {shot:>7} {min(ws):>9.1f}-{max(ws):<11.1f} "
              f"{f'{band[0]}-{band[1]}':>10}  {worst['core']:>6.0%} at "
              f"s={worst['s']:.2f}"
              + (f"   pe refit pinned at {npin}/{len(sub)} points" if npin else ""))

mk = {s: m for s, m in zip(shots, ("o", "s", "^", "D"))}
fig, (axw, axc) = plt.subplots(1, 2, figsize=(13, 4.4))
for shot in shots:
    lo_pe, hi_pe = BOYLE_WIDTHS[camp.regime[shot]]["dpe"]
    axw.axvspan(lo_pe, hi_pe, color="tab:green", alpha=0.10)
for c in checks:
    axw.plot(c["pe_width"], c["s"], mk[c["shot"]], ms=6,
             color=COL.get(AXES[c["axis"]][0], "k"),
             mfc="none" if not c["pe_ok"] else None)
axw.set_xlabel("pe pedestal width  [%psiN]"); axw.set_ylabel("scale factor")
axw.set_title("every check point's pe width vs the Boyle bands\n"
              "(open marker = outside its discharge's band)", fontsize=9)
axw.grid(alpha=0.3)
axw.legend(handles=[plt.Line2D([], [], ls="", marker=mk[s], color="k",
                               label=str(s)) for s in shots], fontsize=6)

for c in checks:
    axc.plot(100 * c["budget"], 100 * c["core"], mk[c["shot"]], ms=6,
             color="tab:red" if c["core"] > c["budget"] else "tab:green")
lim = [0, 1.05 * 100 * max(max(c["core"] for c in checks),
                           max(c["budget"] for c in checks))]
axc.plot(lim, lim, "k--", lw=1.0)
axc.set_xlim(*lim); axc.set_ylim(*lim); axc.grid(alpha=0.3)
axc.set_xlabel("core-drift budget  [%]"); axc.set_ylabel("core drift  [%]")
axc.set_title(f"core drift vs {CORE_FRAC:.0%} of the pedestal-top change\n"
              f"(core = largest fractional change inside rho<{CORE_RHO})",
              fontsize=9)
axc.legend(handles=[plt.Line2D([], [], ls="", marker=mk[s], color="k",
                               label=str(s)) for s in shots], fontsize=6)
plt.tight_layout(); plt.show()

## Handover

`sg_bounds.json` holds the shared box, the ceiling it came from, the discharges
it was validated against, and the Boyle bands for reporting. Next step is
cheaseBS on these profiles — nominals plus the box edges first, since those are
the equilibria most likely to strain the reconstruction, and that run also
settles the reshape-limit question the ±30% ceiling currently stands in for.

Carried forward deliberately, so it is not rediscovered downstream:

- **The box is physical, not literature-matched.** Some discharges cannot reach
  some Boyle bands inside ±30%; that is reported per axis above and does not cut
  the box. The sparse grid explores the box on its own, and Paper 1 reports where
  the sampled points landed in physical units.
- **Core drift is measured and tolerated.** The height axes move the core because
  Stefanikova's core Gaussian is anchored only at $r=0$. If cheaseBS turns out to
  be sensitive to it, the fix is co-scaling `a_height` inside
  `apply_mtanh_full` — a TPED change, not a bounds change.
- **132588's `ne` fit sits on the `b_pos` bound**, so its reported $\Delta n_e$
  (35.7 %$\psi_N$) is not trustworthy and its width response is nonlinear. Its
  profiles are physical, so it stays in the box; only the *reported width* is
  suspect.
- **The $p_e$ width is itself a refit quantity**, so where the refit lands on the
  `b_pos` bound the number is an artefact of the bound rather than a
  measurement — flagged per point above. That is what the wide $p_e$ ranges on
  129038 and 132588 are: not a pedestal swinging from 4% to 29%, but a refit
  losing the pedestal. Nothing downstream depends on it (it is a diagnostic, not
  an axis), but it should not be quoted as physics without checking the flag.